In [1]:
import json  
from datasets import Dataset  

# 加载本地数据  
def load_data(file_path):  
    with open(file_path, "r", encoding="utf-8") as f:  
        data = json.load(f)  
    return Dataset.from_list(data)  

train_dataset = load_data("./pubmed_train.json")  # 训练集  
val_dataset = load_data("./pubmed_val.json")     # 验证集  

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer  
from peft import LoraConfig, get_peft_model  

# 模型名称  
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"  

# 加载 Tokenizer 和预训练模型  
tokenizer = AutoTokenizer.from_pretrained(model_name)  
model = AutoModelForCausalLM.from_pretrained(  
    model_name,  
    device_map="auto"  # 自动分配到 GPU 或 CPU  
)  

# 配置 LoRA 微调  
lora_config = LoraConfig(  
    r=16,                       # LoRA 的秩  
    lora_alpha=32,              # LoRA 的缩放因子  
    target_modules=["q_proj", "v_proj"],  # 仅微调 Query 和 Value 投影部分  
    lora_dropout=0.1,           # Dropout 防止过拟合  
    bias="none",                # 偏置项不进行 LoRA  
    task_type="CAUSAL_LM"       # 因果语言建模任务  
)  

# 将模型转化为 LoRA 模型  
model = get_peft_model(model, lora_config)  
model.print_trainable_parameters()  # 打印可训练参数  

trainable params: 2,179,072 || all params: 1,779,267,072 || trainable%: 0.1225


In [4]:
def preprocess_function(examples):  
    inputs = examples["prompt"]  
    targets = examples["response"]  
    model_inputs = tokenizer(inputs, max_length=256, truncation=True, padding="max_length", return_tensors="pt")  
    labels = tokenizer(targets, max_length=256, truncation=True, padding="max_length", return_tensors="pt")["input_ids"]  
    
    # 将 labels=-100 的地方应用到 padding 部分，避免计算多余的损失  
    labels[labels == tokenizer.pad_token_id] = -100  
    model_inputs["labels"] = labels  
    return model_inputs  

# 应用预处理到数据集  
train_dataset = train_dataset.map(preprocess_function, batched=True)  
val_dataset = val_dataset.map(preprocess_function, batched=True)  

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [5]:
from transformers import TrainingArguments, Trainer  

# 训练参数设置  
training_args = TrainingArguments(  
    output_dir="./results",  
    evaluation_strategy="epoch",     # 每个 epoch 验证  
    save_strategy="epoch",           # 每个 epoch 保存  
    learning_rate=1e-4,              # LoRA 微调的推荐学习率  
    per_device_train_batch_size=4,   # 每张设备训练的 batch size  
    per_device_eval_batch_size=4,    # 每张设备验证的 batch size  
    num_train_epochs=3,              # 总训练 Epoch  
    save_total_limit=2,              # 最多保存的 checkpoint 数  
    logging_dir="./logs",            # 日志输出路径  
    logging_steps=50,                # 每50步打印日志  
    fp16=True,                       # 半精度加速  
    report_to="none"                 # 不上传到 WandB 等服务，默认保存本地  
)  

# 构建 Trainer  
trainer = Trainer(  
    model=model,  
    args=training_args,  
    train_dataset=train_dataset,  
    eval_dataset=val_dataset,  
    tokenizer=tokenizer  
)  

/home/chenkangxin/2TB1/miniconda_envs/LLM/lib/python3.12/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [6]:
trainer.train()  

Epoch,Training Loss,Validation Loss
1,No log,10.314971
2,No log,9.540077
3,11.582400,9.316205


/home/chenkangxin/2TB1/miniconda_envs/LLM/lib/python3.12/site-packages/peft/utils/other.py:716: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/resolve/main/config.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f4cd41615b0>: Failed to establish a new connection: [Errno 101] Network is unreachable'))"), '(Request ID: 692bea63-dd39-4109-8ede-903c11fbdc0f)') - silently ignoring the lookup for the file config.json in deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B.
  warnings.warn(
/home/chenkangxin/2TB1/miniconda_envs/LLM/lib/python3.12/site-packages/peft/utils/save_and_load.py:246: UserWarning: Could not find a config file in deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B - will assume that the vocabulary was not modified.
  warnings.warn(
/home/chenkangxin/2TB1/miniconda_envs/LLM/lib/python3.12/site

TrainOutput(global_step=60, training_loss=11.254673258463542, metrics={'train_runtime': 63.4846, 'train_samples_per_second': 3.78, 'train_steps_per_second': 0.945, 'total_flos': 569878134128640.0, 'train_loss': 11.254673258463542, 'epoch': 3.0})

In [7]:
# 保存模型和分词器  
finetuned_model_path = "./finetuned-DeepSeek"  
model.save_pretrained(finetuned_model_path)  
tokenizer.save_pretrained(finetuned_model_path)  
print(f"模型已保存到 {finetuned_model_path}")  

/home/chenkangxin/2TB1/miniconda_envs/LLM/lib/python3.12/site-packages/peft/utils/other.py:716: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/resolve/main/config.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f4cd42dab70>: Failed to establish a new connection: [Errno 101] Network is unreachable'))"), '(Request ID: a53923a9-0d0c-47b9-b383-17f9493e43fb)') - silently ignoring the lookup for the file config.json in deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B.
  warnings.warn(
/home/chenkangxin/2TB1/miniconda_envs/LLM/lib/python3.12/site-packages/peft/utils/save_and_load.py:246: UserWarning: Could not find a config file in deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B - will assume that the vocabulary was not modified.
  warnings.warn(


模型已保存到 ./finetuned-DeepSeek


In [19]:
from rouge_score import rouge_scorer  

# 使用 rouge_score 库  
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)  

predictions = ["This is a test"]  
references = ["This is a task"]  

scores = scorer.score(predictions[0], references[0])  
print("ROUGE 分数：", scores)  

ROUGE 分数： {'rouge1': Score(precision=0.75, recall=0.75, fmeasure=0.75), 'rouge2': Score(precision=0.6666666666666666, recall=0.6666666666666666, fmeasure=0.6666666666666666), 'rougeL': Score(precision=0.75, recall=0.75, fmeasure=0.75)}


In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM  

# 加载模型和分词器  
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"  
tokenizer = AutoTokenizer.from_pretrained(model_name)  
model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")  

# 设置 `pad_token_id` 为模型的 `eos_token_id`  
if model.config.pad_token_id is None:  
    model.config.pad_token_id = tokenizer.eos_token_id  

# 输入文本  
input_text = "Recent advances in neural network applications for medicine"  

# 分词并输入到 CUDA  
inputs = tokenizer(  
    input_text,  
    return_tensors="pt",  
    padding=True,  
    truncation=True,  
    max_length=128  
).to("cuda")  

# 显式设置 attention_mask 并生成结果  
outputs = model.generate(  
    inputs["input_ids"],  
    attention_mask=inputs["attention_mask"],  # 添加 attention_mask  
    max_length=128,  
    num_beams=5,  
    early_stopping=True  
)  

# 解码并输出结果  
print("生成结果:", tokenizer.decode(outputs[0], skip_special_tokens=True))  

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


生成结果: Recent advances in neural network applications for medicine have shown that neural networks can be used to model the spread of COVID-19 and other infectious diseases. In this problem, we consider a model where the spread of COVID-19 is represented by a graph where each node represents a person, and each edge represents an interaction between two people. It is known that the spread of COVID-19 can be modeled by a graph where each edge has a weight representing the probability of transmission between two people.

Given a graph \( G \) with \( n \) nodes and \( m \) edges, each edge has a weight \( w
